# VGG — depth from stacked 3x3 convolutions

> Tutorial pair for [`vgg.py`](vgg.py).

## 1. Intuition
VGG made one idea uniform and pushed it deep: build the *entire* network from
tiny **3x3 convolutions** (stride 1, pad 1) interleaved with **2x2 max-pools**.
Why small filters? Because a **stack** of them sees just as far as one big filter
(same **receptive field**) while using **fewer parameters** and inserting **more
nonlinearities** (a ReLU between each conv). Uniform, simple, and very deep.

## 2. Concept (the slide)
- **VGG block:** $n$ stacked $3\times3$ conv-ReLU layers (padding keeps spatial
  size), then a $2\times2$ max-pool that halves resolution and doubles channels.
- **Configurations A..E** differ only in how many convs sit in each block — the
  motif is identical.
- **Two $3\times3$ convs $\equiv$ one $5\times5$ receptive field**; three
  $\equiv$ one $7\times7$. The stacked version is cheaper and more expressive.
- **He init** for the ReLU stacks (see `training-techniques/README.md`).

## 3. Math — receptive field & parameter count

**Receptive field of a conv stack.** For layer $l$ with kernel $k_l$ and stride
$s_l$, the receptive field grows as
$$\mathrm{RF}_l = \mathrm{RF}_{l-1} + (k_l-1)\prod_{j<l}s_j .$$
With all strides $=1$ this collapses to
$$\mathrm{RF} = 1 + \sum_l (k_l - 1).$$
So **two** $3\times3$ convs give $\mathrm{RF}=1+2+2=5$ — a $5\times5$ field; **three**
give $7$. A single $7\times7$ conv sees the same window in *one* layer.

**Parameter count.** One conv with $C$ in and $C$ out channels and kernel $k$ has
$k^2C^2$ weights. Compare equal-receptive-field designs:
$$\underbrace{2\,(3^2C^2)}_{\text{two }3\times3}=18C^2 \;<\; \underbrace{5^2C^2}_{\text{one }5\times5}=25C^2,
\qquad
\underbrace{3\,(3^2C^2)}_{\text{three }3\times3}=27C^2 \;<\; \underbrace{7^2C^2}_{\text{one }7\times7}=49C^2.$$
That is $18/25 = 0.72$ (28% fewer) and $27/49 = 0.55$ (45% fewer) parameters —
**and** the stacked form applies a ReLU after every conv, so it represents a
strictly richer (more nonlinear) family of functions for the same field of view.

## 4. Key building block — the VGG 3x3 stack + receptive-field math

In [ ]:
# ===== actual implementation from vgg.py =====
from __future__ import annotations

import numpy as np

SEED = 0

def conv_params(k: int, c_in: int, c_out: int, bias: bool = True) -> int:
    """Weight count of one conv layer: k*k*c_in*c_out (+ c_out biases)."""
    return k * k * c_in * c_out + (c_out if bias else 0)

import torch

import torch.nn as nn

def get_device() -> torch.device:
    if torch.cuda.is_available():
        return torch.device("cuda")
    if torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

VGG_MINI_CFG = [(2, 8), (2, 16)]

def demo():
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    torch.set_num_threads(1)        # tiny ops: avoid thread-thrashing on big CPUs
    dev = get_device()

    # --- (a) Receptive-field & parameter arithmetic: 3x3 stacks vs big convs --
    print("Equal-receptive-field designs (C=64 channels, weights only):")
    for name, (rf, p) in small_vs_large_filter_table(64).items():
        print(f"  {name:10s}: receptive field = {rf}x{rf}, params = {p:>8,}")
    two, one5 = small_vs_large_filter_table(64)["two_3x3"], small_vs_large_filter_table(64)["one_5x5"]
    print(f"  -> two 3x3 match a 5x5 RF with {one5[1] / two[1]:.2f}x FEWER... "
          f"actually {1 - two[1] / one5[1]:.0%} fewer params, + an extra ReLU.")

    # --- (b) Build VGGMini, show shapes & per-layer param counts --------------
    x = torch.randn(8, 1, 16, 16, device=dev)
    yb = torch.randint(0, 10, (8,), device=dev)
    net = VGGMini(in_c=1, n_classes=10).to(dev)
    out = net(x)
    n_params = sum(p.numel() for p in net.parameters())
    print(f"\nVGGMini: input {tuple(x.shape)} -> output {tuple(out.shape)}, "
          f"params = {n_params:,}")
    # trace feature-map shapes block by block
    h = x
    for i, blk in enumerate(net.features):
        h = blk(h)
        print(f"  after block {i}: {tuple(h.shape)}")

    # --- (c) A handful of training steps; loss should drop --------------------
    opt = torch.optim.Adam(net.parameters(), lr=1e-2)
    loss_fn = nn.CrossEntropyLoss()
    losses = []
    for _ in range(20):
        opt.zero_grad()
        loss = loss_fn(net(x), yb)
        loss.backward()
        opt.step()
        losses.append(loss.item())
    print(f"\ntraining loss: {losses[0]:.3f} -> {losses[-1]:.3f} (going down)")


def vgg_block(n_convs: int, in_c: int, out_c: int) -> nn.Sequential:
    """`n_convs` stacked 3x3 conv-ReLU layers (pad 1, so spatial size is kept),
    followed by a 2x2 max-pool that halves the spatial resolution.

    This is the repeating motif of every VGG configuration (A through E differ
    only in how many convs sit in each block)."""
    layers: list[nn.Module] = []
    c = in_c
    for _ in range(n_convs):
        layers.append(nn.Conv2d(c, out_c, kernel_size=3, padding=1))
        layers.append(nn.ReLU(inplace=True))
        c = out_c
    layers.append(nn.MaxPool2d(kernel_size=2, stride=2))
    return nn.Sequential(*layers)


def receptive_field(kernels: list[int], strides: list[int] | None = None) -> int:
    r"""Receptive field of a stack of conv layers (all stride 1 unless given).

    For layer l with kernel k_l and stride s_l, the RF grows as
        RF_l = RF_{l-1} + (k_l - 1) * prod_{j<l} s_j.
    With all strides 1 this is RF = 1 + sum_l (k_l - 1). So two 3x3 convs give
    RF = 1 + 2 + 2 = 5 (a 5x5 field); three give 7.
    """
    if strides is None:
        strides = [1] * len(kernels)
    rf, jump = 1, 1
    for k, s in zip(kernels, strides):
        rf += (k - 1) * jump
        jump *= s
    return rf


def small_vs_large_filter_table(channels: int = 64) -> dict:
    r"""Compare equal-receptive-field designs for C in/out channels (no bias).

    Two 3x3 convs vs one 5x5:    2 * 9 C^2  =  18 C^2   <  25 C^2.
    Three 3x3 convs vs one 7x7:  3 * 9 C^2  =  27 C^2   <  49 C^2.
    The 3x3 stack also inserts an extra ReLU per layer -> more nonlinearity.
    """
    C = channels
    rows = {
        "two_3x3":  (receptive_field([3, 3]),   2 * conv_params(3, C, C, bias=False)),
        "one_5x5":  (receptive_field([5]),          conv_params(5, C, C, bias=False)),
        "three_3x3": (receptive_field([3, 3, 3]), 3 * conv_params(3, C, C, bias=False)),
        "one_7x7":  (receptive_field([7]),          conv_params(7, C, C, bias=False)),
    }
    return rows

## 5. Full architecture (PyTorch) — a config-driven small VGG

In [ ]:
# ===== actual implementation from vgg.py =====
class VGGMini(nn.Module):
    """A miniature VGG: stacked 3x3-conv blocks (each halving resolution via
    max-pool), then a small classifier head. Config-driven exactly like the
    VGG-A..E table in the paper."""

    def __init__(self, cfg=VGG_MINI_CFG, in_c: int = 1, n_classes: int = 10):
        super().__init__()
        blocks: list[nn.Module] = []
        c = in_c
        for n_convs, out_c in cfg:
            blocks.append(vgg_block(n_convs, c, out_c))
            c = out_c
        self.features = nn.Sequential(*blocks)
        self.pool = nn.AdaptiveAvgPool2d(1)        # makes head input-size agnostic
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(c, 32), nn.ReLU(inplace=True),
            nn.Linear(32, n_classes),
        )
        self._he_init()

    def _he_init(self) -> None:
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, nonlinearity="relu")
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.features(x)
        return self.classifier(self.pool(x))

## 6. Run — the 3x3-vs-5x5 table, feature-map shapes, a few training steps

In [ ]:
demo()

## 7. Visualization — parameters vs receptive field: 3x3 stacks win

For each equal-receptive-field design we plot the parameter count. The $3\times3$
stacks (filled markers) sit strictly *below* the single large convs (hollow) at
the same receptive field.

In [ ]:
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
import vgg as M

tbl = M.small_vs_large_filter_table(64)
plt.figure(figsize=(7, 4.2))
for name, (rf, p) in tbl.items():
    stacked = "3x3" in name
    plt.scatter(rf, p, s=140,
                marker="o" if stacked else "s",
                facecolors="C0" if stacked else "none",
                edgecolors="C3" if not stacked else "C0",
                label=name.replace("_", " "))
    plt.annotate(name.replace("_", " "), (rf, p),
                 textcoords="offset points", xytext=(6, 6), fontsize=9)
plt.xlabel("receptive field (pixels)"); plt.ylabel("parameters (C=64, weights)")
plt.title("Same receptive field, fewer params: stacked 3x3 vs one big conv")
plt.grid(True, alpha=.3); plt.tight_layout(); plt.show()

## 8. Takeaways & pitfalls
- **Small filters, stacked deep** = same receptive field, fewer params, more
  nonlinearity. This is the whole VGG thesis.
- **Uniformity** (everything is 3x3/pool) makes VGG a clean baseline, but it is
  **parameter-heavy in the FC head** — most of real VGG-16's 138M params live in
  the dense layers, not the convs. Global average pooling (used here) fixes that.
- **Pitfall:** very deep *plain* stacks still hit the degradation/vanishing-
  gradient wall — VGG was near the practical limit before residual connections
  (ResNet) made arbitrary depth trainable. See `training-techniques/README.md`.